# Fairness Check

did entanglement get a fair chance? better-built circuits and classical models that can combine features, on the best 8 and pls 8.

In [1]:
!pip install -q pennylane==0.45.1

import os
os.environ["OMP_NUM_THREADS"] = "1"
import multiprocessing as mp
import numpy as np
import pandas as pd
import torch
import pennylane as qml
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

seed = 0
np.random.seed(seed)
torch.set_num_threads(1)

prep_dir = "/kaggle/input/datasets/malakkhaled202300369/nic-feature-dataset/prep"
clip_dir = "/kaggle/input/datasets/malakkhaled202300369/nic-clip-dataset/features"
out_dir = "/kaggle/working/fairness"
os.makedirs(out_dir, exist_ok=True)

seeds = [0]
lr = 0.01
epochs = 5
batch_size = 256
c_grid = [0.01, 0.1, 1.0]                        # sin/cos twin
gb_grid = [(0.05, 15), (0.1, 31), (0.1, 63)]     # gradient boosting (learning rate, max leaf nodes)
alpha_grid = [1e-4, 1e-3, 1e-2]                  # mlp 64x64, about 4800 weights on 8 inputs

In [2]:
angles = {}
meta = {}
for split in ["train", "val"]:
    angles[split] = np.load(f"{prep_dir}/ffpp_{split}_angles32.npy").astype(np.float64)
    meta[split] = pd.read_csv(f"{prep_dir}/ffpp_{split}_rows.csv")

is_inner = np.load(f"{prep_dir}/ffpp_train_is_inner.npy").astype(bool)
comp = pd.read_csv(f"{prep_dir}/per_component_auc.csv")
best8 = sorted(comp.sort_values("strength_search_train", ascending=False)["component"].head(8).tolist())
first8 = list(range(8))

y_train = meta["train"]["label"].to_numpy()
y_val = meta["val"]["label"].to_numpy()
# a video is (method, video_id), the 4 fakes of one pair share a video_id
val_video = meta["val"].groupby(["method", "video_id"]).ngroup().to_numpy()

## the two inputs

best 8 from the per-component auc, and 8 pls components fitted on search-train only and scaled with search-train percentiles.

In [3]:
clip = {}
for split in ["train", "val"]:
    clip_rows = pd.read_csv(f"{clip_dir}/ffpp_{split}_rows.csv")
    assert (clip_rows["path"].to_numpy() == meta[split]["path"].to_numpy()).all(), split
    x = np.load(f"{clip_dir}/ffpp_{split}_clip.npy").astype(np.float64)
    clip[split] = x / np.linalg.norm(x, axis=1, keepdims=True)
st = ~is_inner

pls = PLSRegression(n_components=8, scale=False)
pls.fit(clip["train"][st], y_train[st])
z = {split: pls.transform(clip[split]) for split in clip}
lo = np.percentile(z["train"][st], 0.5, axis=0)
hi = np.percentile(z["train"][st], 99.5, axis=0)

inputs = {
    "best 8": {split: angles[split][:, best8] for split in angles},
    "PLS 8": {split: np.clip((z[split] - lo) / (hi - lo), 0.0, 1.0) * np.pi for split in z},
}
def video_level(prob):
    df = pd.DataFrame({"video": val_video, "p": prob, "y": y_val})
    g = df.groupby("video").agg(p=("p", "mean"), y=("y", "first"))
    return g["p"].to_numpy(), g["y"].to_numpy()


def bootstrap_ci(p, y, n=1000, seed=0):
    rng = np.random.default_rng(seed)
    aucs = []
    while len(aucs) < n:
        idx = rng.integers(0, len(y), len(y))
        if y[idx].min() == y[idx].max():
            continue
        aucs.append(roc_auc_score(y[idx], p[idx]))
    return np.percentile(aucs, [2.5, 97.5])


def score(prob):
    pv, yv = video_level(prob)
    lo, hi = bootstrap_ci(pv, yv)
    return {"frame auc": roc_auc_score(y_val, prob), "video auc": roc_auc_score(yv, pv), "ci low": lo, "ci high": hi}


def sincos(a):
    return np.concatenate([np.sin(a), np.cos(a)], axis=1)


def predict(model, x, batch_size=4096):
    out = []
    with torch.no_grad():
        for s in range(0, len(x), batch_size):
            xb = torch.as_tensor(x[s:s + batch_size], dtype=torch.float64)
            out.append(torch.sigmoid(model(xb)))
    return torch.cat(out).numpy()

## classical models

3 settings each, picked on val.

In [4]:
def make_model(kind, setting):
    if kind == "sin/cos twin":
        return make_pipeline(StandardScaler(), LogisticRegression(C=setting, max_iter=5000))
    if kind == "gradient boosting":
        return HistGradientBoostingClassifier(learning_rate=setting[0], max_leaf_nodes=setting[1], max_iter=300,
                                              early_stopping=True, random_state=0)
    return make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(64, 64), alpha=setting, max_iter=200,
                                                         early_stopping=True, random_state=0))


grids = {"sin/cos twin": c_grid, "gradient boosting": gb_grid, "mlp 64x64": alpha_grid}
table = []
for name, data in inputs.items():
    for kind, grid in grids.items():
        xt = data["train"]
        xv = data["val"]
        if kind == "sin/cos twin":
            xt = sincos(xt)
            xv = sincos(xv)
        best = None
        for setting in grid:
            row = score(make_model(kind, setting).fit(xt, y_train).predict_proba(xv)[:, 1])
            if best is None or row["video auc"] > best["video auc"]:
                best = dict(row, input=name, family="classical", model=kind, setting=str(setting))
        table.append(best)

## circuits with adjustable entanglers

ry encoding and rot rotations as in the grid. a circuit is its list of entanglers, one per layer. `crz_ring` is a ring of crz gates whose 8 angles are trained, starting at 0 where the gate does nothing.

In [5]:
n_qubits = 8


def build_circuit(layers):
    dev = qml.device("default.qubit", wires=n_qubits)

    @qml.qnode(dev, interface="torch", diff_method="backprop")
    def circuit(x, ws, ent_ws):
        for i in range(n_qubits):
            qml.RY(x[:, i], wires=i)
        for ent, w, ew in zip(layers, ws, ent_ws):
            for i in range(n_qubits):
                qml.Rot(w[i, 0], w[i, 1], w[i, 2], wires=i)
            if ent == "chain":
                for i in range(n_qubits - 1):
                    qml.CNOT(wires=[i, i + 1])
            elif ent == "ring":
                for i in range(n_qubits):
                    qml.CNOT(wires=[i, (i + 1) % n_qubits])
            elif ent == "crz_ring":
                for i in range(n_qubits):
                    qml.CRZ(ew[i], wires=[i, (i + 1) % n_qubits])
        return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

    return circuit


class QuantumClassifier(torch.nn.Module):
    def __init__(self, layers):
        super().__init__()
        self.circuit = build_circuit(layers)
        self.rot = torch.nn.ParameterList()
        self.ent = torch.nn.ParameterList()
        for ent in layers:
            self.rot.append(torch.nn.Parameter(0.1 * torch.randn(n_qubits, 3, dtype=torch.float64)))
            trainable = ent == "crz_ring"
            self.ent.append(torch.nn.Parameter(torch.zeros(n_qubits, dtype=torch.float64), requires_grad=trainable))
        self.readout = torch.nn.Linear(n_qubits, 1, dtype=torch.float64)
        torch.nn.init.normal_(self.readout.weight, std=0.1)
        torch.nn.init.zeros_(self.readout.bias)

    def expvals(self, x):
        return torch.stack(self.circuit(x, list(self.rot), list(self.ent)), dim=1)

    def forward(self, x):
        return self.readout(self.expvals(x)).squeeze(1)


def train(model, x, y, seed=0):
    g = torch.Generator().manual_seed(seed)
    x = torch.as_tensor(x, dtype=torch.float64)
    y = torch.as_tensor(y, dtype=torch.float64)
    opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
    loss_fn = torch.nn.BCEWithLogitsLoss()
    for epoch in range(epochs):
        order = torch.randperm(len(x), generator=g)
        for start in range(0, len(x), batch_size):
            idx = order[start:start + batch_size]
            opt.zero_grad()
            loss_fn(model(x[idx]), y[idx]).backward()
            opt.step()


circuits = {
    "additive, depth 1": ["none"],
    "ring then rotation": ["ring", "none"],
    "chain then rotation": ["chain", "none"],
    "2 rings then rotation": ["ring", "ring", "none"],
    "adjustable ring then rotation": ["crz_ring", "none"],
    "2 adjustable rings then rotation": ["crz_ring", "crz_ring", "none"],
}

# with the angles at 0 the adjustable circuit must read the same as the one without entanglers
torch.manual_seed(0)
xb = torch.as_tensor(inputs["best 8"]["train"][:50], dtype=torch.float64)
adjustable = QuantumClassifier(["crz_ring", "none"])
plain = QuantumClassifier(["none", "none"])
with torch.no_grad():
    for p_plain, p_adj in zip(plain.rot, adjustable.rot):
        p_plain.copy_(p_adj)
    assert torch.allclose(adjustable.expvals(xb), plain.expvals(xb), atol=1e-10)

In [6]:
def run_task(task):
    name, label, s = task
    layers = circuits[label]
    torch.manual_seed(s)
    model = QuantumClassifier(layers)
    train(model, inputs[name]["train"], y_train, seed=s)
    row = {"input": name, "family": "circuit", "model": label, "seed": s}
    row.update(score(predict(model, inputs[name]["val"])))
    # how far training moved the adjustable angles from 0, in radians
    moved = []
    for w, ent in zip(model.ent, layers):
        if ent == "crz_ring":
            moved.append(float(w.detach().abs().mean()))
    if len(moved) > 0:
        row["crz angle"] = np.mean(moved)
    return row


tasks = []
for i in inputs:
    for c in circuits:
        for s in seeds:
            tasks.append((i, c, s))
with mp.get_context("fork").Pool(os.cpu_count()) as pool:
    circuit_runs = pool.map(run_task, tasks)

## results

differences under about 0.02 are inside the noise of one val run.

In [7]:
pd.DataFrame(circuit_runs).to_csv(f"{out_dir}/circuit_runs.csv", index=False)
results = pd.DataFrame(table + circuit_runs)
results = results.groupby(["input", "family", "model"], sort=False).agg(
    video_auc=("video auc", "mean"), ci_low=("ci low", "min"), ci_high=("ci high", "max"),
    frame_auc=("frame auc", "mean"), crz_angle=("crz angle", "mean")).reset_index()
results.to_csv(f"{out_dir}/fairness_results.csv", index=False)

for name in inputs:
    t = results[results["input"] == name].set_index("model")
    print(name)
    print(t.drop(columns=["input"]).round(3).to_string())
    twin = t.loc["sin/cos twin", "video_auc"]
    nonlinear = t.loc[["gradient boosting", "mlp 64x64"], "video_auc"].max()
    additive = t.loc["additive, depth 1", "video_auc"]
    entangled = t[(t["family"] == "circuit") & (t.index != "additive, depth 1")]["video_auc"].max()
    print(f"nonlinear minus twin {nonlinear - twin:+.3f}, entangled minus additive {entangled - additive:+.3f}\n")

best 8
                                     family  video_auc  ci_low  ci_high  frame_auc  crz_angle
model                                                                                        
sin/cos twin                      classical      0.713   0.667    0.754      0.691        NaN
gradient boosting                 classical      0.714   0.670    0.755      0.688        NaN
mlp 64x64                         classical      0.697   0.651    0.740      0.671        NaN
additive, depth 1                   circuit      0.712   0.667    0.753      0.691        NaN
ring then rotation                  circuit      0.701   0.653    0.744      0.680        NaN
chain then rotation                 circuit      0.702   0.655    0.745      0.681        NaN
2 rings then rotation               circuit      0.709   0.665    0.748      0.687        NaN
adjustable ring then rotation       circuit      0.712   0.666    0.752      0.690      0.990
2 adjustable rings then rotation    circuit      0.71